In [1]:
## 完整 SDK 使用示例
# 前提：在项目目录运行 `langgraph dev` 启动本地服务器
from langgraph_sdk import get_client
from langchain_core.messages import HumanMessage

# 1. 连接到本地服务器
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 2. 搜索可用的助手
assistants = await client.assistants.search()
print(f"找到 {len(assistants)} 个助手")
print(f"助手 ID: {assistants[0]['graph_id']}")

# 3. 创建对话线程
thread = await client.threads.create()
print(f"线程 ID: {thread['thread_id']}")

# 4. 运行图并流式输出
input_data = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# 确保在运行前重启一下 langgraph dev 且不再改动文件
print("\n=== 开始运行 (updates 模式) ===")
# 清理旧的线程数据，重新创建一个
thread = await client.threads.create()

print("\n=== 增量流式输出 ===")
async for chunk in client.runs.stream(
    thread['thread_id'],
    assistants[0]['graph_id'],
    input=input_data,
    stream_mode="updates", # 这里改掉
):
    # 过滤掉不含数据的 metadata 事件
    if chunk.event == "metadata" or not chunk.data:
        continue
    
    # updates 模式的数据结构是 { "节点名": { "state_key": value } }
    for node_name, data in chunk.data.items():
        if "messages" in data:
            new_msg = data["messages"][-1]
            # 获取内容
            content = new_msg.get('content') if isinstance(new_msg, dict) else getattr(new_msg, 'content', '')
            print(f"\n[节点: {node_name}] 类型: {new_msg.get('type')}")
            if content:
                print(f"内容: {content}")
            
            # 检查是否有工具调用
            t_calls = new_msg.get('tool_calls', [])
            if t_calls:
                print(f"🛠️ 工具调用: {t_calls[0]['name']}({t_calls[0]['args']})")

找到 1 个助手
助手 ID: agent
线程 ID: 019dee24-c8b3-7f40-8ce4-5181787ffd36

=== 开始运行 (updates 模式) ===

=== 增量流式输出 ===

[节点: assistant] 类型: ai
🛠️ 工具调用: multiply({'a': 3, 'b': 2})

[节点: tools] 类型: tool
内容: 6

[节点: assistant] 类型: ai
内容: The result of multiplying 3 by 2 is **6**.


In [2]:
## 完整 SDK 使用示例
# 前提：在项目目录运行 `langgraph dev` 启动本地服务器
from langgraph_sdk import get_client
from langchain_core.messages import HumanMessage

# 1. 连接到本地服务器
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 2. 搜索可用的助手
assistants = await client.assistants.search()
print(f"找到 {len(assistants)} 个助手")
print(f"助手 ID: {assistants[0]['graph_id']}")

# 3. 创建对话线程
thread = await client.threads.create()
print(f"线程 ID: {thread['thread_id']}")

# 4. 运行图并流式输出
input_data = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

print("\n=== 流式输出 ===")
async for chunk in client.runs.stream(
    thread['thread_id'],
    assistants[0]['graph_id'],
    input=input_data,
    stream_mode="values",
):
    if chunk.data and chunk.event != "metadata":
        last_msg = chunk.data['messages'][-1]
        print(f"[{last_msg.get('type', 'unknown')}] {last_msg.get('content', '')[:100]}")

找到 1 个助手
助手 ID: agent
线程 ID: 019dee25-1f03-74e0-915b-c1ed0b5b685b

=== 流式输出 ===
[human] Multiply 3 by 2.
[ai] 
[tool] 6
[ai] The result of multiplying 3 by 2 is **6**.
